# ToysAD-8K DS0 Prompted SAM 2.1 Segmentation

**File:** `05-toysad8k-ds0-sam21-segmentation.ipynb`

This notebook generates one SAM 2.1 mask for every frozen object prompt in the complete DS0 cohort. It runs the reference-box and LocateAnything-box routes under the same model configuration.

This stage tests operational coverage, repeatability, runtime and failure handling. SAM confidence is recorded as model evidence, not as ground-truth mask accuracy. Mask IoU, Dice and boundary quality are calculated later in the evaluator-only stage.

## Run contract

- 20 frozen DS0 scenes
- 20 RGB views per scene
- 400 images
- 1,640 reference-box prompts
- 1,616 automatic-box prompts
- one-image preflight before the pilot
- 20-image pilot covering every scene and view position
- exact repeat check before the full run
- 3,256 mask outcomes in the completed run

The notebook stops before the full run if the preflight or repeat gate fails. Model-facing inputs contain no anomaly labels, categories, supplied masks or source identifiers.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

from IPython.display import HTML, Image as NotebookImage, display


NOTEBOOK_NAME = "05_toysad8k_ds0_sam21_segmentation.ipynb"
OUTPUT_ROOT = Path("/kaggle/working/toysad8k_ds0_sam21_segmentation")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"

## Locate and verify the frozen inputs

In [ ]:
def locate_directory(marker: str) -> Path:
    candidates: list[Path] = []
    marker_path = Path(marker)
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in input_root.rglob(marker_path.name):
            if path.parts[-len(marker_path.parts) :] == marker_path.parts:
                candidates.append(path.parents[len(marker_path.parts) - 1])
    candidates.extend((Path.cwd(), *Path.cwd().parents))
    for candidate in candidates:
        if (candidate / marker).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate an input containing {marker}")


CODE_ROOT = locate_directory("src/visual_peer_inspector/workflows/sam_ds0.py")
MODEL_INPUT_ROOT = locate_directory("neutral_corpus_manifest.jsonl")
PROMPT_ROOT = locate_directory("sam_prompt_manifest.jsonl")
MEDIA_ROOT = MODEL_INPUT_ROOT / "media"

sys.path.insert(0, str(CODE_ROOT / "src"))

from visual_peer_inspector.data.model_safe_package import file_sha256
from visual_peer_inspector.data.sam_prompt_package import verify_sam_prompt_package
from visual_peer_inspector.verification import verify_code_snapshot_record


snapshot = verify_code_snapshot_record(CODE_ROOT, CODE_ROOT / "snapshot_record.json")
prompt_record = verify_sam_prompt_package(PROMPT_ROOT)

if prompt_record["image_count"] != 400:
    raise ValueError("The SAM prompt package does not contain 400 images")
if prompt_record["reference_prompt_count"] != 1640:
    raise ValueError("The reference prompt count is not 1,640")
if prompt_record["automatic_prompt_count"] != 1616:
    raise ValueError("The automatic prompt count is not 1,616")

input_summary = {
    "code_files": snapshot["file_count"],
    "code_snapshot_sha256": snapshot["snapshot_sha256"],
    "git_commit": snapshot["git_commit"],
    "precommit_candidate": snapshot["precommit_candidate"],
    "images": prompt_record["image_count"],
    "reference_prompts": prompt_record["reference_prompt_count"],
    "automatic_prompts": prompt_record["automatic_prompt_count"],
    "prompt_package_sha256": file_sha256(PROMPT_ROOT / "package_record.json"),
}
display(input_summary)

## Prepare the pinned runtime

In [ ]:
install_started = time.perf_counter()
model_load_checkpoints: list[dict] = [
    {"stage": "dependency_install_started", "elapsed_seconds": 0.0}
]

install_environment = os.environ.copy()
install_environment["SAM2_BUILD_CUDA"] = "0"
print(json.dumps({"stage": "sam_dependency_install_started"}), flush=True)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/sam2-lock.txt")],
    env=install_environment,
    timeout=300,
)
print(json.dumps({"stage": "sam_source_install_started"}), flush=True)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4",
    ],
    env=install_environment,
    timeout=600,
)
print(json.dumps({"stage": "sam_runtime_ready"}), flush=True)

model_load_checkpoints.append(
    {
        "stage": "dependency_install_completed",
        "elapsed_seconds": round(time.perf_counter() - install_started, 3),
    }
)

In [ ]:
from visual_peer_inspector.models.sam2 import Sam2Configuration


configuration = Sam2Configuration()
distribution = importlib.metadata.distribution("SAM-2")
direct_url = json.loads(distribution.read_text("direct_url.json") or "{}")
installed_revision = direct_url.get("vcs_info", {}).get("commit_id")
installed_repository = str(direct_url.get("url", "")).removesuffix(".git")

if installed_revision != configuration.code_revision:
    raise RuntimeError("The installed SAM source revision is not the pinned revision")
if installed_repository != configuration.code_repository:
    raise RuntimeError("The installed SAM repository is not the pinned repository")

model_load_checkpoints.append(
    {
        "stage": "sam_installation_verified",
        "elapsed_seconds": round(time.perf_counter() - install_started, 3),
    }
)
display(
    {
        "package": distribution.metadata["Name"],
        "package_version": distribution.version,
        "repository": installed_repository,
        "revision": installed_revision,
    }
)

## Verify the GPU and model checkpoint

In [ ]:
import torch
from huggingface_hub import hf_hub_download


if not torch.cuda.is_available():
    raise RuntimeError("Notebook 05 requires a Kaggle GPU")

device_index = 0
device = torch.cuda.get_device_properties(device_index)
compute_capability = f"{device.major}.{device.minor}"
supported_architectures = set(torch.cuda.get_arch_list())
architecture = f"sm_{device.major}{device.minor}"
gpu_evidence = {
    "cuda_available": True,
    "device_count": torch.cuda.device_count(),
    "device_name": torch.cuda.get_device_name(device_index),
    "compute_capability": compute_capability,
    "total_memory_gib": round(device.total_memory / (1024**3), 3),
    "torch_architecture": architecture,
    "torch_architecture_supported": architecture in supported_architectures,
}
if not gpu_evidence["torch_architecture_supported"]:
    raise RuntimeError("The installed PyTorch build does not support this GPU architecture")

model_load_checkpoints.append(
    {
        "stage": "checkpoint_download_started",
        "elapsed_seconds": round(time.perf_counter() - install_started, 3),
    }
)
download_started = time.perf_counter()
checkpoint_path = Path(
    hf_hub_download(
        repo_id=configuration.model_repository,
        filename=configuration.checkpoint_filename,
        revision=configuration.model_revision,
        cache_dir="/tmp/vpi-sam21-model-cache",
    )
)
checkpoint_size = checkpoint_path.stat().st_size
checkpoint_sha256 = file_sha256(checkpoint_path)
if checkpoint_size != configuration.checkpoint_size_bytes:
    raise RuntimeError("The downloaded SAM checkpoint size is incorrect")
if checkpoint_sha256 != configuration.checkpoint_sha256:
    raise RuntimeError("The downloaded SAM checkpoint hash is incorrect")

model_load_checkpoints.append(
    {
        "stage": "checkpoint_download_completed",
        "elapsed_seconds": round(time.perf_counter() - install_started, 3),
        "download_seconds": round(time.perf_counter() - download_started, 3),
        "checkpoint_size_bytes": checkpoint_size,
        "checkpoint_sha256": checkpoint_sha256,
    }
)
display({**gpu_evidence, "checkpoint_size_bytes": checkpoint_size, "checkpoint_sha256": checkpoint_sha256})

## Generate the masks

The workflow uses one GPU even if Kaggle exposes two. Each image embedding is calculated once, then all reference and automatic boxes for that image are passed as a batch. Completed image records are atomic and can be resumed without repeating successful work.

In [ ]:
from visual_peer_inspector.workflows.sam_ds0 import run_ds0_sam_workflow


notebook_path = CODE_ROOT / "notebooks" / NOTEBOOK_NAME
snapshot_versions = snapshot.get("platform_versions", {})
required_version_keys = (
    "kaggle_notebook_version_05",
    "code_dataset_version",
    "prompt_dataset_version",
    "model_input_dataset_version",
)
missing_version_keys = [key for key in required_version_keys if key not in snapshot_versions]
if missing_version_keys:
    raise RuntimeError(f"The code snapshot is missing version pins: {missing_version_keys}")
provenance = {
    "git_commit": snapshot["git_commit"],
    "git_dirty": snapshot["git_dirty"],
    "snapshot_sha256": snapshot["snapshot_sha256"],
    "precommit_candidate": snapshot["precommit_candidate"],
    "notebook_sha256": file_sha256(notebook_path),
    "kaggle_notebook_slug": "esreekarreddy/toysad-8k-ds0-sam-2-1-segmentation",
    "kaggle_notebook_version": str(snapshot_versions["kaggle_notebook_version_05"]),
    "code_dataset_version": str(snapshot_versions["code_dataset_version"]),
    "prompt_dataset_version": str(snapshot_versions["prompt_dataset_version"]),
    "model_input_dataset_version": str(snapshot_versions["model_input_dataset_version"]),
}
platform_versions = {
    "python": platform.python_version(),
    "numpy": importlib.metadata.version("numpy"),
    "pillow": importlib.metadata.version("pillow"),
    "torch": importlib.metadata.version("torch"),
    "torchvision": importlib.metadata.version("torchvision"),
    "kaggle_notebook_version": provenance["kaggle_notebook_version"],
    "code_dataset_version": provenance["code_dataset_version"],
    "prompt_dataset_version": provenance["prompt_dataset_version"],
    "model_input_dataset_version": provenance["model_input_dataset_version"],
}


def report_progress(stage: str, details: dict) -> None:
    if stage in {"sam_import_started", "sam_model_build_started", "sam_worker_ready"}:
        if not any(row["stage"] == stage for row in model_load_checkpoints):
            model_load_checkpoints.append(
                {
                    "stage": stage,
                    "elapsed_seconds": round(time.perf_counter() - install_started, 3),
                }
            )
    print(json.dumps({"stage": stage, **details}, sort_keys=True), flush=True)


run_record = run_ds0_sam_workflow(
    prompt_root=PROMPT_ROOT,
    media_root=MEDIA_ROOT,
    output_root=OUTPUT_ROOT,
    configuration=configuration,
    run_mode="model_inference",
    inference_stage="full",
    checkpoint_path=checkpoint_path,
    checkpoint_sha256=checkpoint_sha256,
    provenance=provenance,
    platform_versions=platform_versions,
    gpu_evidence=gpu_evidence,
    model_load_rows=model_load_checkpoints,
    progress_callback=report_progress,
)

if not run_record["completion_gate_passed"]:
    raise RuntimeError("The SAM 2.1 completion gate failed")
display(run_record)

## Operational evidence

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


safe_root = OUTPUT_ROOT / "safe"
completion_gates = pd.read_csv(safe_root / "completion_gates.csv")
route_summary = pd.read_csv(safe_root / "route_summary.csv")
view_summary = pd.read_csv(safe_root / "view_summary.csv")

display(HTML("<h4>Completion gates</h4>"))
display(completion_gates)
display(HTML("<h4>Route summary</h4>"))
display(route_summary)
display(HTML("<h4>View summary</h4>"))
display(view_summary)

chart_specs = [
    ("runtime_distribution.png", "Runtime"),
    ("runtime_by_prompt_count.png", "Runtime by prompt count"),
    ("mask_count_distribution.png", "Masks per image"),
    ("mask_count_by_route.png", "Masks by route"),
    ("reference_box_mask_area_distribution.png", "Reference-box mask area"),
    ("automatic_box_mask_area_distribution.png", "Automatic-box mask area"),
    ("reference_box_predicted_score_distribution.png", "Reference-box quality score"),
    ("automatic_box_predicted_score_distribution.png", "Automatic-box quality score"),
]
fig, axes = plt.subplots(2, 4, figsize=(18, 8.4))
for axis, name, title in zip(
    axes.flat,
    [item[0] for item in chart_specs],
    [item[1] for item in chart_specs],
):
    axis.imshow(plt.imread(safe_root / name))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

## Qualitative SAM mask examples

The panel below uses six deterministic pilot views. Each view is shown as RGB, SAM prompted by the reference boxes, and SAM prompted by the frozen LocateAnything boxes. Colours identify separate object masks and the matching outline shows the prompt box.

These examples confirm that the model generated object masks and allow direct visual inspection of both prompt routes. They do not measure ground-truth accuracy. That comparison is performed in Notebook 06.

In [ ]:
private_root = OUTPUT_ROOT / "private"
route_example_path = private_root / "mask_route_examples.png"
if not route_example_path.is_file():
    raise FileNotFoundError("The qualitative SAM route panel was not generated")
display(NotebookImage(filename=str(route_example_path)))

## Record the aggregate evidence in ClearML

Tracking starts only after inference and artifact verification. The finalizer accepts aggregate tables, charts and hashes only. It rejects masks, boxes, images, identifiers and evaluator fields. A timeout cannot erase the completed mask package.

In [ ]:
def prepare_tracking_environment() -> dict[str, str]:
    environment = os.environ.copy()
    existing_pythonpath = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = str(CODE_ROOT / "src") + (
        os.pathsep + existing_pythonpath if existing_pythonpath else ""
    )
    if EXECUTION_ENVIRONMENT == "kaggle":
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
            value = client.get_secret(name)
            if not value:
                raise RuntimeError(f"Kaggle Secret {name} is missing or disabled")
            environment[name] = value
    environment.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
    environment.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
    environment.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
    environment.setdefault("CLEARML_LOG_ENVIRONMENT", "")
    return environment


tracking_receipt = OUTPUT_ROOT / "tracking_receipt.json"
tracking_command = [
    sys.executable,
    str(CODE_ROOT / "scripts/finalize_sam_clearml.py"),
    "--safe-root",
    str(safe_root),
    "--receipt",
    str(tracking_receipt),
    "--mode",
    "online",
    "--tag",
    "candidate",
    "--tag",
    "notebook-05",
    "--tag",
    "ds0",
    "--tag",
    "sam2.1",
    "--tag",
    "kaggle",
    "--tag",
    "full",
]

try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "clearml==2.1.10"],
        text=True,
        capture_output=True,
        timeout=180,
        check=True,
    )
    completed_tracking = subprocess.run(
        tracking_command,
        env=prepare_tracking_environment(),
        text=True,
        capture_output=True,
        timeout=180,
        check=True,
    )
    tracking_result = json.loads(tracking_receipt.read_text(encoding="utf-8"))
except Exception as tracking_error:
    tracking_result = {
        "schema_version": 1,
        "status": "failed",
        "tracking_mode": "online",
        "clearml_task_id": None,
        "error_type": type(tracking_error).__name__,
    }
    tracking_receipt.write_text(
        json.dumps(tracking_result, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

display(tracking_result)

## Final result

In [ ]:
final_summary = {
    "status": run_record["status"],
    "completion_gate_passed": run_record["completion_gate_passed"],
    "images": run_record["summary"]["image_count"],
    "masks": run_record["summary"]["mask_count"],
    "empty_masks": run_record["summary"]["empty_mask_count"],
    "repeat_mask_agreement": run_record["repeat_agreement"]["mask_digest_agreement_rate"],
    "repeat_score_agreement": run_record["repeat_agreement"]["score_agreement_rate"],
    "median_seconds_per_image": run_record["summary"]["median_image_runtime_seconds"],
    "p95_seconds_per_image": run_record["summary"]["p95_image_runtime_seconds"],
    "peak_gpu_memory_gib": run_record["gpu"].get("peak_allocated_gib"),
    "tracking_status": tracking_result["status"],
    "clearml_task_id": tracking_result.get("clearml_task_id"),
}
display(final_summary)

if not run_record["completion_gate_passed"]:
    raise RuntimeError("Notebook 05 did not complete its operational contract")